In [2]:
from typing import TypedDict,Annotated
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from langgraph.graph import START,END,StateGraph
from langgraph.checkpoint.memory import MemorySaver

In [3]:
class AgentState(TypedDict):
    messages:Annotated[list[AnyMessage],add_messages]

In [4]:
def agent_node(state:AgentState):
    return {
        "messages":[
            {
                "role":"assistant",
                "content":f"You said {state['messages'][-1].content}"
            }
        ]
    }

In [5]:
checkpointer = MemorySaver()
graph=StateGraph(AgentState)
graph.add_node("agent",agent_node)
graph.add_edge(START,"agent")
graph.add_edge("agent",END)  
app=graph.compile(checkpointer=checkpointer)

In [6]:
config={
    "configurable":{
        "thread_id":"user_1"
    }
}
result =app.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"My name is Priyanshu"  
            }
        ]
    },
    config=config

)

result

{'messages': [HumanMessage(content='My name is Priyanshu', additional_kwargs={}, response_metadata={}, id='e97c4a24-b632-4ee1-b04a-241dc80a9f3d'),
  AIMessage(content='You said My name is Priyanshu', additional_kwargs={}, response_metadata={}, id='91577f69-1aec-4262-a1cc-57c462a77137', tool_calls=[], invalid_tool_calls=[])]}

In [7]:
ans=app.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"What is my name"
            }
        ]
    },   
    config=config
)

ans

{'messages': [HumanMessage(content='My name is Priyanshu', additional_kwargs={}, response_metadata={}, id='e97c4a24-b632-4ee1-b04a-241dc80a9f3d'),
  AIMessage(content='You said My name is Priyanshu', additional_kwargs={}, response_metadata={}, id='91577f69-1aec-4262-a1cc-57c462a77137', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is my name', additional_kwargs={}, response_metadata={}, id='8f8e453b-923d-4b47-ac7c-759fa3d7fbf6'),
  AIMessage(content='You said What is my name', additional_kwargs={}, response_metadata={}, id='ebe554f4-79f8-4f89-9174-1687aa889d88', tool_calls=[], invalid_tool_calls=[])]}

In [8]:
new_config = {
    "configurable": {
        "thread_id": "user_2"
    }
}

result = app.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is my name?"
            }
        ]
    },
    config=new_config
)

print(result)

{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='44991031-fbd0-4e9f-9d9d-27cb1019ad56'), AIMessage(content='You said What is my name?', additional_kwargs={}, response_metadata={}, id='30286755-a48d-4f6d-953e-8498a09230b4', tool_calls=[], invalid_tool_calls=[])]}


In [9]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
conn = sqlite3.connect(
    "checkpoints.db",
    check_same_thread=False
)

checkpointer = SqliteSaver(conn)

app = graph.compile(checkpointer=checkpointer)

In [10]:
config = {
    "configurable": {
        "thread_id": "user_1"
    }
}

result = app.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Do you remember my name?"
            }
        ]
    },
    config=config
)

print(result)

{'messages': [HumanMessage(content='Do you remember my name?', additional_kwargs={}, response_metadata={}, id='902dc40f-f5b6-41e3-8d48-a8926e1d875e'), AIMessage(content='You said Do you remember my name?', additional_kwargs={}, response_metadata={}, id='46409d6d-5aee-4296-a964-53defb9e95fa', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Do you remember my name?', additional_kwargs={}, response_metadata={}, id='078a8f16-0e5e-4693-8c40-4ecda370367e'), AIMessage(content='You said Do you remember my name?', additional_kwargs={}, response_metadata={}, id='23437205-e06d-4af2-be88-6a57cebee602', tool_calls=[], invalid_tool_calls=[])]}


In [11]:
config = {
    "configurable": {
        "thread_id": "persistent_test"
    }
}
app.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My name is Priyanshu"
            }
        ]
    },
    config=config
)

{'messages': [HumanMessage(content='My name is Priyanshu', additional_kwargs={}, response_metadata={}, id='abd667e5-24b2-4eb2-b89f-cc0b705437d1'),
  AIMessage(content='You said My name is Priyanshu', additional_kwargs={}, response_metadata={}, id='bc0ff368-2574-4cfd-b1b1-bf57ee08a8d0', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='My name is Priyanshu', additional_kwargs={}, response_metadata={}, id='33fb1cb5-0ca9-4266-a60f-48a3c88a2330'),
  AIMessage(content='You said My name is Priyanshu', additional_kwargs={}, response_metadata={}, id='4b4a2daa-9737-4f67-b27d-db3f45767a10', tool_calls=[], invalid_tool_calls=[])]}

In [12]:
state = app.get_state(config)

print(state.values)

{'messages': [HumanMessage(content='My name is Priyanshu', additional_kwargs={}, response_metadata={}, id='abd667e5-24b2-4eb2-b89f-cc0b705437d1'), AIMessage(content='You said My name is Priyanshu', additional_kwargs={}, response_metadata={}, id='bc0ff368-2574-4cfd-b1b1-bf57ee08a8d0', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='My name is Priyanshu', additional_kwargs={}, response_metadata={}, id='33fb1cb5-0ca9-4266-a60f-48a3c88a2330'), AIMessage(content='You said My name is Priyanshu', additional_kwargs={}, response_metadata={}, id='4b4a2daa-9737-4f67-b27d-db3f45767a10', tool_calls=[], invalid_tool_calls=[])]}
